In [1]:
from pathlib import Path
import pandas as pd
import re

# ========= 改成你的路徑 =========
input_csv = Path(r"/Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/health_tables/ethnicity_health_shares.csv")
output_tex = Path(r"/Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/health_tables/ethnicity_health_shares.tex")
# ===============================

def latex_escape(text):
    if pd.isna(text):
        return ""
    text = str(text)
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    pattern = re.compile("|".join(re.escape(k) for k in replacements))
    return pattern.sub(lambda m: replacements[m.group(0)], text)

df = pd.read_csv(input_csv)

cols = [
    "ethn_group",
    "p_covidsymp",
    "p_resp",
    "p_cardio",
    "p_endo",
    "p_arth",
    "p_other",
    "n",
]
missing = [c for c in cols if c not in df.columns]
if missing:
    raise ValueError(f"CSV 缺少欄位: {missing}")

df = df[cols].copy()

num_cols = [
    "p_covidsymp",
    "p_resp",
    "p_cardio",
    "p_endo",
    "p_arth",
    "p_other",
    "n",
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

ethnicity_order = [
    "British/English/Scottish/Welsh/Northern Irish",
    "Indian",
    "Pakistani",
    "Bangladeshi",
    "African",
    "Caribbean",
    "Any other white background",
]
df["ethn_group"] = pd.Categorical(
    df["ethn_group"],
    categories=ethnicity_order,
    ordered=True
)
df = df.sort_values("ethn_group").reset_index(drop=True)

df["ethn_group"] = df["ethn_group"].astype(str).map(latex_escape)

lines = []
lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(r"\caption{Health Conditions by Ethnicity}")
lines.append(r"\label{tab:ethnicity_health}")
lines.append(r"\begin{threeparttable}")
lines.append(r"\footnotesize")
lines.append(r"\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}p{5.2cm}ccccccc}")
lines.append(r"\toprule")
lines.append(r"& \multicolumn{1}{c}{COVID} & \multicolumn{5}{c}{Chronic conditions} & \\")
lines.append(r"\cmidrule(lr){2-2}\cmidrule(lr){3-7}")
lines.append(r"Ethnicity & Symptoms & Respiratory & Cardiovascular & Endocrine & Arthritis & Other & N \\")
lines.append(r"\midrule")

for _, row in df.iterrows():
    eth = row["ethn_group"]
    covid = "" if pd.isna(row["p_covidsymp"]) else f'{row["p_covidsymp"]:.2f}'
    resp  = "" if pd.isna(row["p_resp"]) else f'{row["p_resp"]:.2f}'
    card  = "" if pd.isna(row["p_cardio"]) else f'{row["p_cardio"]:.2f}'
    endo  = "" if pd.isna(row["p_endo"]) else f'{row["p_endo"]:.2f}'
    arth  = "" if pd.isna(row["p_arth"]) else f'{row["p_arth"]:.2f}'
    other = "" if pd.isna(row["p_other"]) else f'{row["p_other"]:.2f}'
    n     = "" if pd.isna(row["n"]) else f'{int(round(row["n"])):,}'
    lines.append(f"{eth} & {covid} & {resp} & {card} & {endo} & {arth} & {other} & {n} \\\\")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular*}")
lines.append(r"\begin{tablenotes}[flushleft]")
lines.append(r"\footnotesize")
lines.append(
    r"\item Notes: This table reports weighted percentages of COVID symptoms and selected health conditions by ethnicity. Respiratory, Cardiovascular, Endocrine, Arthritis, and Other refer to the corresponding chronic-condition indicators. Percentages are weighted using the CA Covid survey weights, and \(N\) denotes the unweighted sample size."
)
lines.append(r"\end{tablenotes}")
lines.append(r"\end{threeparttable}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)
output_tex.write_text(latex_table, encoding="utf-8")

print(f"LaTeX table saved to: {output_tex}")
print()
print(latex_table)

LaTeX table saved to: /Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/health_tables/ethnicity_health_shares.tex

\begin{table}[htbp]
\centering
\caption{Health Conditions by Ethnicity}
\label{tab:ethnicity_health}
\begin{threeparttable}
\footnotesize
\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}p{5.2cm}ccccccc}
\toprule
& \multicolumn{1}{c}{COVID} & \multicolumn{5}{c}{Chronic conditions} & \\
\cmidrule(lr){2-2}\cmidrule(lr){3-7}
Ethnicity & Symptoms & Respiratory & Cardiovascular & Endocrine & Arthritis & Other & N \\
\midrule
British/English/Scottish/Welsh/Northern Irish & 11.38 & 16.03 & 20.56 & 10.07 & 12.37 & 21.93 & 12,534 \\
Indian & 13.92 & 8.94 & 8.34 & 18.94 & 6.24 & 13.90 & 433 \\
Pakistani & 9.12 & 10.25 & 4.40 & 6.20 & 3.94 & 4.90 & 265 \\
Bangladeshi & 19.46 & 7.72 & 9.35 & 11.94 & 1.02 & 3.58 & 100 \\
African & 30.23 & 9.14 & 16.64 & 8.10 & 6.54 & 32.46 & 112 \\
Caribbean & 8.58 & 16.95 & 26.08 & 15.43 & 36.60 & 27.01 & 131 \\
Any other white